In [1]:
## import pandas
import pandas as pd

In [19]:
## import data
sales_df = pd.read_excel('3_AC Property Sale Transactions_20260701.xlsx',dtype={'DEEDBOOK':object,
                                                                                'DEEDPAGE':object,
                                                                                'PROPERTYHOUSENUM':object})

In [28]:
sales_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495583 entries, 0 to 495582
Data columns (total 24 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   parid                    495583 non-null  object        
 1   propertyhousenum         495576 non-null  object        
 2   propertyfraction         495583 non-null  object        
 3   propertyaddressdir       20549 non-null   object        
 4   propertyaddressstreet    495567 non-null  object        
 5   propertyaddresssuf       493526 non-null  object        
 6   propertyaddressunitdesc  12030 non-null   object        
 7   propertyunitno           11652 non-null   object        
 8   propertycity             495578 non-null  object        
 9   propertystate            495583 non-null  object        
 10  propertyzip              495582 non-null  float64       
 11  schoolcode               495583 non-null  int64         
 12  schooldesc      

In [21]:
## import property assessment data
assessed_df = pd.read_excel('1_AC Property Assessments_20260714.xlsx',dtype={'PROPERTYHOUSENUM':object,
'PROPERTYFRACTION':object,
'PROPERTYZIP':object,
'MUNICODE':object,
'TAXYEAR':object,
'SCHOOLCODE':object,
'NEIGHCODE':object,
'TAXCODE':object,
'OWNERCODE':object,
'USECODE':object,
'SALECODE':object,
})

In [30]:
sales_df.parid.nunique()

309247

## Clean

In [23]:
## make column names lowercase
sales_df.columns = sales_df.columns.str.lower()
assessed_df.columns = assessed_df.columns.str.lower()

In [35]:
## strip columns of whitespace
sales_df['deedbook'] = sales_df['deedbook'].str.strip()
sales_df['deedpage'] = sales_df['deedpage'].str.strip()
sales_df['propertycity'] = sales_df['propertycity'].str.strip()

assessed_df['deedbook'] = assessed_df['deedbook'].str.strip()
assessed_df['deedpage'] = assessed_df['deedpage'].str.strip()
assessed_df['propertycity'] = assessed_df['propertycity'].str.strip()


In [54]:
## change dtype
assessed_df['saledate'] = assessed_df['saledate'].astype('datetime64[ns]')

In [66]:
## just choosing stuff sold in 2025
sales_2025 = sales_df[sales_df['saledate'] >= '2025-01-01']

In [65]:
sales_2025

,parid,propertyhousenum,propertyfraction,propertyaddressdir,propertyaddressstreet,propertyaddresssuf,propertyaddressunitdesc,propertyunitno,propertycity,propertystate,...,munidesc,recorddate,saledate,price,deedbook,deedpage,salecode,saledesc,instrtyp,instrtypdesc
53,0003K00094000000,0,,NaN,ARLINGTON,AVE,NaN,NaN,PITTSBURGH,PA,...,17th Ward - PITTSBURGH,2026-04-30 00:00:00,2025-11-17,1.0,NaN,NaN,H,MULTI-PARCEL SALE,QC,QUIT CLAIM
113,0943G00224000000,0,,NaN,HARMONY,RD,NaN,NaN,PITTSBURGH,PA,...,McCandless,2025-12-29 00:00:00,2025-12-23,75000.0,NaN,NaN,H,MULTI-PARCEL SALE,DE,DEED
146,0552C00175000000,0,,NaN,RAINBOW,AVE,NaN,NaN,MC KEESPORT,PA,...,White Oak,2025-09-26 00:00:00,2025-09-22,50000.0,NaN,NaN,99,CORRECTIVE DEED / DUPLICATE SALE,CO,CORRECTIVE DEED
77759,1519S00240000000,0,,NaN,KUNTZ,ST,NaN,NaN,NATRONA HEIGHTS,PA,...,Harrison,2026-05-04 00:00:00,2026-04-17,1.0,NaN,NaN,3,LOVE AND AFFECTION SALE,DE,DEED
235435,0547S00224000000,1006,,NaN,JACOB,ST,NaN,NaN,NORTH VERSAILLES,PA,...,North Versailles,2025-01-28 00:00:00,2025-01-17,10000.0,NaN,NaN,37,ESTATE SALE,DE,DEED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495578,0062P00163000000,422,,NaN,BROOKLINE,BLVD,NaN,NaN,PITTSBURGH,PA,...,19th Ward - PITTSBURGH,2025-10-29 00:00:00,2025-10-28,100000.0,NaN,NaN,14,TIME ON MARKET (INSUFF/EXCESS),GW,GENERAL WARRANTY
495579,0098M00256000000,3076,,NaN,DELWOOD,AVE,NaN,NaN,PITTSBURGH,PA,...,Dormont,2025-10-28 00:00:00,2025-10-28,1.0,NaN,NaN,3,LOVE AND AFFECTION SALE,DE,DEED
495580,0129G00122000000,1146,,NaN,LOVE,ST,NaN,NaN,PITTSBURGH,PA,...,14th Ward - PITTSBURGH,2025-10-22 00:00:00,2025-09-10,1.0,NaN,NaN,3,LOVE AND AFFECTION SALE,QC,QUIT CLAIM
495581,0112E00259000000,29,,NaN,COLES,ROW,NaN,NaN,MC KEES ROCKS,PA,...,Stowe,2025-10-22 00:00:00,2025-10-21,10.0,NaN,NaN,3,LOVE AND AFFECTION SALE,DE,DEED


## Merge

Notes: 
- 309,247 unique parcels sold last year. But there are 495583 rows. So some parcels were sold multiple times in one year. 

In [55]:
merged_df = pd.merge(sales_df,
                     assessed_df,
                     on=['parid','deedbook','deedpage','propertycity','propertyhousenum','saledate'],
                     how='left')

In [56]:
merged_df.parid.nunique()

309247

In [57]:
len(merged_df)

495583

In [59]:
dupes_df = merged_df[merged_df.duplicated('parid', keep=False) == True]

In [63]:
test = dupes_df[dupes_df['parid'] == '1075F00108000000']
test.to_csv('dupe_test.csv')

## Analyze

In [58]:
merged_df.propertyowner.nunique()

243313

In [41]:
merged_df.groupby('propertyowner')['parid'].nunique().sort_values(ascending=False).head(20)

propertyowner
CITY OF PITTSBURGH                            3582
REDEVELOPMENT AUTHORITY OF ALLEGHENYCOUNTY     372
PENNSYLVANIA TURNPIKE COMMISSION               302
URBAN REDEVELOPMENT AUTHORITY OFPITTSBURGH     217
ALLEGHENY LAND TRUST                           201
SEGAVEPO                                       185
CLAIRTON COMMUNITY PROPERTIES LLC              177
BOROUGH OF WILKINSBURG                         175
PEOPLES NATURAL GAS COMPANY LLC                169
MS CAPITAL GROUP PITTSBURGH LLC                162
CORONADO IV LLC                                156
IMPERIAL LAND PROPERTIES LP                    132
VB ONE LLC                                     125
NORTHSIDE PROPERTIES RESIDENCES II LLC         110
CASTLEROCK PROPERTY LLC                        110
MARONDA HOMES, LLC                             105
THREE RIVERS COMMUNITIES INC                   104
ALLEGHENY COUNTY SANITARY AUTHORITY             98
WHITEHALL PLACE HOLDINGS LLC                    97
SJ GROUP         